# 07.1 — comparing residual→goal probes on `W_OV`

Same prompt setup as `7_ov_reader_eval_demo`. For a residual `h` (after layer `L` at a pivot) we compare
five ways to turn `h` into a goal-steering direction, then read each out through GoalScope:

1. **GoalScope skyline** — the goal heads' actual OUTPUT read-out (`apply_goalscope`). Upper reference.
2. **OV(h)** — the transform `W_OV·h = W_O W_V diag(g) h` (current OVReader approach).
3. **write-proj** — `h` projected onto the **column/write** space of `W_OV`, `W_O(W_O⁺h)`.
   *(Provably = projection onto colspace(W_O) — the old O-write probe — since `W_V·diag(g)` is full row-rank.)*
4. **read-proj** — `h` projected onto the **row/read** space of `W_OV`, `B⁺(Bh)` with `B=W_V diag(g)`.
5. **eigen** — `h` projected onto the top-`EIG_TOPK` eigenspace of `W_OV` (the OV-loop's self-consistent
   directions; `W_OV` is an endomorphism so eigenvectors live in the residual space).

Each variant makes a per-head `d`-vector, unit-normalized and averaged over the goal heads → one steering
direction, fed to `head_additive_generate` (additive_norm).

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os, sys
os.environ["CUDA_VISIBLE_DEVICES"] = "5, 6"   # pick a GPU with ~62GB free (avoid the eval jobs)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
sys.path.append(".."); sys.path.append("../..")

import torch, baukit
from tqdm.auto import tqdm
from rich import print as rich_print
from rich.markup import escape
from rich.table import Table

import scripts.evaluation.eval_ov_reader as R
import src.diff.functional as P
from scripts.evaluation.eval_goalscope import hard_eval, CAUSAL_PREFIX
from src.goals.goalscope import (apply_goalscope, greedy_continue, head_additive_generate,
                                 _build_goalscope_prompt, _build_causality_prompt)
from src.functional import get_hs
from src.tokens import prepare_input
from src.models import ModelandTokenizer

os.chdir(os.path.abspath(os.path.join(os.path.dirname(R.__file__), "..", "..")))

# ── config ──
MODEL         = "Qwen/Qwen2.5-Coder-32B-Instruct"
HEADS_JSON    = R.DEFAULT_HEADS_JSON.format(slug=MODEL.split("/")[-1])
torch.set_grad_enabled(False)

torch.autograd.grad_mode.set_grad_enabled(mode=False)

In [3]:
mt = ModelandTokenizer(model_key=MODEL, dtype=torch.bfloat16,
                       device_map="auto", attn_implementation="eager")
HEADS = R.load_json_heads(HEADS_JSON, mt.n_layer, mt.model.config.num_attention_heads)
GSP = _build_causality_prompt(mt) if MODE == "causality" else None
PROBE_INPUTS = prepare_input(prompts=(GSP or _build_goalscope_prompt(mt)), tokenizer=mt)
PREFIX = CAUSAL_PREFIX if MODE == "causality" else ""
rich_print(f"heads={len(HEADS)}  layers={LAYERS}  pivot={PIVOT}  mode={MODE}  {DATATYPE}/{LANG}")

2026-09-09 12:59:12 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

heads=122  layers=[36, 40, 44, 48, 52, 56]  pivot=eq  mode=faithfulness  int/python

In [4]:
# ── Probes: cache per-head factors of W_OV = W_O W_V diag(g); expose the 4 residual-side variants ──
class Probes:
    """Per goal head, cache (W_O, B=W_V·diag(g)) and their pinvs + the top-k eigenspace basis Q of
    W_OV. Each `vecs(h)` entry aggregates unit(per-head vector) over heads -> one steering direction."""
    def __init__(self, mt, heads, eig_topk=64):
        hd = mt.head_dim; nkv = mt.n_kv_heads; group = mt.n_heads // nkv
        self.R = []
        for (l, j) in tqdm(list(heads), desc="building probe factors"):
            kv = j // group
            Wo = baukit.get_module(mt._model, P._proj_name(l, "o", mt)).weight[:, j*hd:(j+1)*hd].detach().float().cpu()   # (d,hd)
            Wv = baukit.get_module(mt._model, P._proj_name(l, "v", mt)).weight[kv*hd:(kv+1)*hd, :].detach().float().cpu() # (hd,d)
            g  = baukit.get_module(mt._model, mt.layer_name_format.format(l) + ".input_layernorm").weight.detach().float().cpu()
            B  = Wv * g[None, :]                       # (hd,d) = W_V diag(g)
            Wo_pinv = torch.linalg.pinv(Wo)            # (hd,d)
            B_pinv  = torch.linalg.pinv(B)             # (d,hd)
            # eigenvectors of W_OV via the 128x128 companion C=B@Wo (shares nonzero eigenpairs):
            #   W_OV (Wo w) = Wo B Wo w = Wo (C w) = lambda (Wo w)  -> eigvec x = Wo w
            C = B @ Wo                                 # (hd,hd)
            lam, w = torch.linalg.eig(C)
            idx = torch.argsort(lam.abs(), descending=True)[:eig_topk]
            Xc = Wo.to(torch.complex64) @ w[:, idx]    # (d,k) complex eigenvectors of W_OV
            X = torch.cat([Xc.real, Xc.imag], dim=1)   # real invariant subspace spanned by the pair
            Q, _ = torch.linalg.qr(X)                  # (d, <=2k) orthonormal basis of the eigenspace
            self.R.append(dict(Wo=Wo, B=B, Wo_pinv=Wo_pinv, B_pinv=B_pinv, Q=Q))

    @staticmethod
    def _agg(vs):
        return torch.stack([v / v.norm().clamp_min(1e-8) for v in vs]).mean(0)

    def vecs(self, h):
        h = h.detach().float().cpu()
        ov, wp, rp, eg = [], [], [], []
        for r in self.R:
            ov.append(r["Wo"] @ (r["B"] @ h))          # OV(h): transform W_OV h
            wp.append(r["Wo"] @ (r["Wo_pinv"] @ h))    # write/col-space projection (= O-write)
            rp.append(r["B_pinv"] @ (r["B"] @ h))      # read/row-space projection
            eg.append(r["Q"] @ (r["Q"].T @ h))         # eigenspace projection
        return {"OV(h)": self._agg(ov), "write-proj": self._agg(wp),
                "read-proj": self._agg(rp), "eigen": self._agg(eg)}

probes = Probes(mt, HEADS, eig_topk=EIG_TOPK)
APPROACHES = ["OV(h)", "write-proj", "read-proj", "eigen"]

building probe factors:   0%|          | 0/122 [00:00<?, ?it/s]

In [45]:
DATATYPE      = "list[str]"           # int | str | list[int] | list[str]
LANG          = "python"
MODE          = "faithfulness"  # faithfulness | causality
N_EXAMPLES    = 1
PIVOT         = "eq"            # prev_end | indent | var | eq | first_impl
LAYERS        = [36, 40, 44, 48, 52, 56]
STEER         = (35, 46)
ALPHA         = 0.15            # residual-probe steering
SKYLINE_ALPHA = 0.1
MAXNEW        = 64
GEN_NEW       = 64
SEED          = 321
EIG_TOPK      = 64             # eigenvectors of W_OV kept for the eigen-projection (<= d_head=128)

In [46]:
# ── build N_EXAMPLES datapoints (same as nb7) ──
from scripts.evaluation import ov_reader_io as ovio
datapoints = R.sample_datapoints(DATATYPE, [3], 64, LANG, SEED)[:N_EXAMPLES]
examples = []
for i, inst in enumerate(datapoints):
    prompt, var = R.build_prompt(mt, inst, LANG)
    completion = greedy_continue(mt, **prepare_input(prompts=prompt, tokenizer=mt), max_new_tokens=GEN_NEW)
    full_text = prompt + completion
    rich_print(f"[white]{prompt}[/white][sea_green3]{completion}[/sea_green3]")
    pivs = R.pivot_positions(mt, prompt, full_text, ovio.PIVOT_ORDER)
    examples.append(dict(spec=f"{DATATYPE}_{LANG}_{i:03d}", full_text=full_text, pivots=pivs,
                         impl_span=R.frontier_impl_span(prompt, completion, var),
                         frontier=inst.expected_next_desc))
for e in examples:
    rich_print(f"[bold]{e['spec']}[/]  frontier=[green]{escape(e['frontier'])}[/]  "
               f"pivot {PIVOT}@{e['pivots'].get(PIVOT)}")

<|im_start|>system
You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just the 
code, no need to comment or explain.<|im_end|>
<|im_start|>user
Write a Python program that implements a function `func` which, given a list of strings `words`, computes in 
sequential order:
(1) whether any word is empty
(2) the number of words longer than five characters
(3) the number of words that start with a vowel
<|im_end|>
<|im_start|>assistant
<code>
def func(words: list):
    a = any(len(w) == 0 for w in words)
    b = sum(1 for w in words if len(w) > 5)
    c = sum(1 for w in words if w.lower().startswith(('a', 'e', 'i', 'o', 'u')))
    return a, b, c
</code>

list_python_000  frontier=the number of words that start with a vowel  pivot eq@150

In [47]:
# ── (1) READOUTS: generate OV/proj/eigen steering read-outs across LAYERS + the skyline, per example ──
# batched over layers (one head_additive_generate per approach); progress bars + per-example logging.
RO = {}
for e in tqdm(examples, desc="readouts · examples"):
    pos = e["pivots"].get(PIVOT)
    if pos is None:
        tqdm.write(f"[skip] {e['spec']}: pivot {PIVOT!r} not found"); continue
    # residuals after each probe layer at the pivot (one forward), then per-layer steering vecs
    locs = [(mt.layer_name_format.format(L), pos) for L in LAYERS]
    hs = get_hs(mt, prepare_input(prompts=e["full_text"], tokenizer=mt), locations=locs, return_dict=True)
    vbyL = {L: probes.vecs(hs[(mt.layer_name_format.format(L), pos)]) for L in LAYERS}
    mats = {a: [vbyL[L][a] for L in LAYERS] for a in APPROACHES}
    readouts = {}
    for a in tqdm(APPROACHES, desc=f"{e['spec']} · gen", leave=False):
        outs = head_additive_generate(mt, PROBE_INPUTS, torch.stack(mats[a]),
                    layers=list(range(*STEER)), max_new_tokens=MAXNEW,
                    strategy="additive_norm", alpha=ALPHA)
        readouts[a] = {L: PREFIX + ro for L, ro in zip(LAYERS, outs)}
    sky = apply_goalscope(e["full_text"], mt=mt, selected_heads=HEADS, strategy="additive_norm",
                          steer_layer_range=STEER, alpha=SKYLINE_ALPHA, n_steer_tokens="all",
                          max_new_tokens=MAXNEW, show=False, include_noncode=False,
                          token_positions=[pos], goalscope_prompt=GSP)
    RO[e["spec"]] = dict(pos=pos, impl_span=e["impl_span"], frontier=e["frontier"],
                         readouts=readouts, skyline=PREFIX + sky[0]["patched_gen"])
    tqdm.write(f"[readouts] {e['spec']}: {len(APPROACHES)} approaches x {len(LAYERS)} layers + skyline "
               f"(pivot {PIVOT}@{pos})")
print(f"done - read-outs for {len(RO)} example(s), {len(APPROACHES)} approaches x {len(LAYERS)} layers each")

readouts · examples:   0%|          | 0/1 [00:00<?, ?it/s]

list[str]_python_000 · gen:   0%|          | 0/4 [00:00<?, ?it/s]

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

[readouts] list[str]_python_000: 4 approaches x 6 layers + skyline (pivot eq@150)
done - read-outs for 1 example(s), 4 approaches x 6 layers each


In [48]:
# ── (2) EVAL: grade every read-out with the oracle (hard_eval); progress bar over all grading calls ──
tasks = []                                            # (spec, approach|"skyline", layer|None, readout)
for spec, r in RO.items():
    for a in APPROACHES:
        for L in LAYERS:
            tasks.append((spec, a, L, r["readouts"][a][L]))
    tasks.append((spec, "skyline", None, r["skyline"]))

EV = {spec: {**{a: {} for a in APPROACHES}, "skyline": None} for spec in RO}
for spec, key, L, ro in tqdm(tasks, desc="eval · oracle"):
    v = hard_eval(ro, RO[spec]["impl_span"], MODE)["verdict"]
    if key == "skyline":
        EV[spec]["skyline"] = v
    else:
        EV[spec][key][L] = v
print(f"done - graded {len(tasks)} read-outs "
      f"({len(RO)} examples x [{len(APPROACHES)}x{len(LAYERS)} + 1 skyline])")

eval · oracle:   0%|          | 0/25 [00:00<?, ?it/s]

done - graded 25 read-outs (1 examples x [4x6 + 1 skyline])


In [53]:
# ── (3) PRINT: per-example table (rows = layer, cols = approach; coloured verdicts) + skyline ──
VC = {"YES": "bold green3", "NO": "bold red3", "NO_INFO": "yellow3",
      "HEAVY_LOOP": "magenta3", "MALFORMED": "grey50", "UNPARSED": "grey50"}
def vtag(v): return f"[{VC.get(v, 'white')}]{(v or '?')[:3]}[/]"
def trunc(s, n=82): return (s[:n] + "…") if s and len(s) > n else (s or "")

for spec, r in RO.items():
    t = Table(title=f"{spec} · pivot={PIVOT} · frontier: {r['frontier']}",
              title_style="bold", header_style="bold", border_style="grey42")
    t.add_column("layer", no_wrap=True)
    for a in APPROACHES:
        t.add_column(a, overflow="fold")
    for L in LAYERS:
        row = [str(L)]
        for a in APPROACHES:
            row.append(f"{vtag(EV[spec][a][L])} {escape(trunc(r['readouts'][a][L]))}")
        t.add_row(*row)
    rich_print(t)
    rich_print(f"  [bold]skyline (goalscope)[/] {vtag(EV[spec]['skyline'])} "
               f"[italic]{escape(trunc(r['skyline'], 80))}[/]\n")

                list_python_000 · pivot=eq · frontier: the number of words that start with a vowel                 
┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ layer ┃ OV(h)                   ┃ write-proj               ┃ read-proj               ┃ eigen                    ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 36    │ NO I'm counting         │ NO_ I'm processing the   │ NO_ I'm not sure what   │ NO_ I am processing the  │
│       │ occurrences of a        │ line of code to          │ specific line you're    │ line to identify         │
│       │ specific character in a │ determine its            │ referring to, but it    │ specific operations or   │
│       │ string.                 │ functionality.           │ would be the count of … │ operations involving u…  │
│ 40    │ YES I'm checking if the │ NO_ I'm checking if a    │ YES I'm checking if the │ NO This line checks      │
│       │ character is a vowel.   │ specific condition is    │ current character is a  │ whether the variable is  │
│       │                         │ met in the code.         │ vowel (assuming the     │ a boolean ( checking if  │
│       │                         │                          │ line is `sum(1 for …    │ it is a boolean o…       │
│ 44    │ YES I'm checking if the │ NO_ I'm checking if the  │ YES I'm assuming you're │ NO_ I am processing the  │
│       │ variable starts with a  │ variable `x` is being    │ referring to a line of  │ line checking for        │
│       │ vowel (a, e, i, o, u)   │ used to store values     │ code where a variable   │ specific patterns in the │
│       │ using a simple co…      │ that match a specif…     │ is initialized wi…      │ code, identifying w…     │
│ 48    │ YES I'm checking for    │ YES I'm checking for     │ YES Sure, please        │ HEA I'm counting how     │
│       │ occurrences of          │ occurrences of specific  │ provide the line of     │ many times the variable  │
│       │ characters in the       │ characters in the        │ code and I'll count how │ has been incremented,    │
│       │ string that match 'a',  │ string, counting how ma… │ many times it matches   │ using a loop for ea…     │
│       │ 'e', 'i',…              │                          │ cer…                    │                          │
│ 52    │ YES Sure, I need to     │ YES I'm counting the     │ YES Sure, I can help    │ YES I'm counting the     │
│       │ know which specific     │ number of vowels in a    │ with that. Please       │ number of occurrences of │
│       │ characters or patterns  │ string, specifically for │ provide the specific    │ specific characters in   │
│       │ to look for in your     │ the vowels 'a', 'e…      │ line of code you want   │ the string, consi…       │
│       │ tex…                    │                          │ me t…                   │                          │
│ 56    │ NO Sure, please specify │ NO_ I'm processing the   │ YES Sure, I'll count    │ YES I'm assuming you're  │
│       │ the exact characters or │ line of code that        │ the number of lines     │ referring to a specific  │
│       │ keywords that should be │ involves checking for    │ that start with a       │ line of code, but you    │
│       │ considered fo…          │ specific conditions rel… │ specific letter in your │ didn't specify w…        │
│       │                         │                          │ cod…                    │                          │
└───────┴─────────────────────────┴──────────────────────────┴─────────────────────────┴──────────────────────────┘

skyline (goalscope) YES Checking if the character is a vowel.

In [51]:
# ── (3b) FULL read-outs (untruncated) — per example, grouped by approach ──
_VC = {"YES": "bold green3", "NO": "bold red3", "NO_INFO": "yellow3",
       "HEAVY_LOOP": "magenta3", "MALFORMED": "grey50", "UNPARSED": "grey50"}
def _tag(v): return f"[{_VC.get(v, 'white')}]{(v or '?')[:3]}[/]"

for spec, r in RO.items():
    rich_print(f"[bold]══ {spec} · pivot={PIVOT} · frontier: {escape(r['frontier'])} ══[/]")
    rich_print(f"  [dim]impl_span:[/] {escape(r['impl_span'])}")
    rich_print(f"  [bold]skyline (goalscope)[/] {_tag(EV[spec]['skyline'])}  [italic]{escape(r['skyline'])}[/]")
    for a in APPROACHES:
        rich_print(f"  [bold underline]{a}[/]")
        for L in LAYERS:
            rich_print(f"    L={L:>2} {_tag(EV[spec][a][L])}  [italic][white]{escape(r['readouts'][a][L])}[/][/]")
    rich_print("")

══ list_python_000 · pivot=eq · frontier: the number of words that start with a vowel ══

impl_span: c = sum(1 for w in words if w.lower().startswith(('a', 'e', 'i', 'o', 'u')))

skyline (goalscope) YES  Checking if the character is a vowel.

OV(h)

L=36 NO  I'm counting occurrences of a specific character in a string.

L=40 YES  I'm checking if the character is a vowel.

... (21 outputs left out)

L=44 NO_  I am processing the line checking for specific patterns in the code, identifying whether it involves 
counting or checking for specific characters.

L=48 HEA  I'm counting how many times the variable has been incremented, using a loop for each iteration that 
matches the number of occurrences in the string.
```python
for t in "ae"
for s in "ae"
for s in "ae"
for s in "ae"
for s in "ae"
for s in

L=52 YES  I'm counting the number of occurrences of specific characters in the string, considering only vowels 
(vowels like 'a', 'e', ' o' in the string, and so on.

L=56 YES  I'm assuming you're referring to a specific line of code, but you didn't specify which line or the 
context. It's common to count lines of code that match a certain pattern, often including the specified line. For 
example, if you're counting lines that start with a specific character, like 'a' or